In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import pickle
import os

# Load data and model
df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')

with open('C:/Users/ELITE/Documents/AGROALERT/src_model/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Encode
le_community = LabelEncoder()
le_region = LabelEncoder()
df['community_enc'] = le_community.fit_transform(df['community'])
df['region_enc'] = le_region.fit_transform(df['region'])

features = ['community_enc', 'region_enc', 'ndvi', 'rainfall_mm',
            'temp_max', 'temp_min', 'humidity', 'et0',
            'lst_celsius', 'ndvi_anomaly', 'rainfall_deficit',
            'water_balance', 'spei_proxy']

X = df[features]

# Get RF probability scores
df['rf_score'] = rf_model.predict_proba(X)[:, 1]

# Simple rule-based score as second component
# (acts as our LSTM substitute for now)
df['rule_score'] = 0.0
df.loc[df['spei_proxy'] < -1.0, 'rule_score'] += 0.4
df.loc[df['ndvi_anomaly'] < -0.05, 'rule_score'] += 0.3
df.loc[df['rainfall_deficit'] < -10, 'rule_score'] += 0.2
df.loc[df['lst_celsius'] > 38, 'rule_score'] += 0.1

# Ensemble: weighted combination
df['ensemble_score'] = (0.6 * df['rf_score']) + (0.4 * df['rule_score'])

# Apply alert threshold
df['alert_triggered'] = (df['ensemble_score'] >= 0.65).astype(int)

print("Ensemble scores calculated!")
print(f"\nAlert summary:")
print(df.groupby('community')['alert_triggered'].sum())
print(f"\nTop drought risk records:")
print(df[df['ensemble_score'] > 0.3][
    ['community', 'date', 'ndvi', 'rainfall_mm', 
     'spei_proxy', 'ensemble_score', 'alert_triggered']
].sort_values('ensemble_score', ascending=False).head(10).to_string())

# Save predictions
df.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv', index=False)
print("\nPredictions saved!")

Ensemble scores calculated!

Alert summary:
community
Bolgatanga          1
Cape Coast          0
Dambai              1
Damongo             3
Goaso               4
Ho                  4
Koforidua           4
Kumasi              1
Nalerigu            1
Sefwi Wiawso        0
Sekondi-Takoradi    1
Sunyani             3
Tamale              1
Techiman            2
Wa                  0
Name: alert_triggered, dtype: int64

Top drought risk records:
       community        date      ndvi  rainfall_mm  spei_proxy  ensemble_score  alert_triggered
479        Goaso  2023-02-19  0.355434          0.6   -1.171674        0.957013                1
61    Bolgatanga  2023-03-05  0.104269          0.0   -1.191359        0.910943                1
1211     Sunyani  2023-01-29  0.207547          9.4   -1.594664        0.878986                1
361      Damongo  2022-11-20  0.512593          0.1   -1.037702        0.878830                1
422        Goaso  2022-01-16  0.366173          0.0   -1.143988     

In [2]:
import shutil

# Copy dashboard to your project
src = r'C:\Users\ELITE\Documents\AGROALERT\src_dashboard\dashboard.html'

html_code = open(src, 'w', encoding='utf-8')
# We'll write it directly
print("Ready - paste the HTML into the file")

Ready - paste the HTML into the file


In [3]:
import pandas as pd
import numpy as np
import pickle

df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')

with open('C:/Users/ELITE/Documents/AGROALERT/src_model/rf_model.pkl','rb') as f:
    rf_model = pickle.load(f)
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/le_community.pkl','rb') as f:
    le_community = pickle.load(f)
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/le_region.pkl','rb') as f:
    le_region = pickle.load(f)

df['community_enc'] = le_community.transform(df['community'])
df['region_enc'] = le_region.transform(df['region'])

features = ['community_enc','region_enc','ndvi','rainfall_mm',
            'temp_max','temp_min','humidity','et0',
            'lst_celsius','ndvi_anomaly','rainfall_deficit',
            'water_balance','spei_proxy']

df['rf_score'] = rf_model.predict_proba(df[features])[:,1]

# Rule-based score
df['rule_score'] = 0.0
df.loc[df['spei_proxy'] < -1.0, 'rule_score'] += 0.4
df.loc[df['ndvi_anomaly'] < -0.05, 'rule_score'] += 0.3
df.loc[df['rainfall_deficit'] < -10, 'rule_score'] += 0.2
df.loc[df['lst_celsius'] > 38, 'rule_score'] += 0.1

# Ensemble
df['ensemble_score'] = (0.6 * df['rf_score']) + (0.4 * df['rule_score'])
df['alert_triggered'] = (df['ensemble_score'] >= 0.65).astype(int)

df.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv', index=False)

print("=== AgroAlert Ghana — 15 Community Alert Summary ===\n")
summary = df.groupby('community').agg(
    alerts=('alert_triggered','sum'),
    max_score=('ensemble_score','max'),
    region=('region','first')
).sort_values('max_score', ascending=False)

for community, row in summary.iterrows():
    score = row['max_score']
    status = "🔴 HIGH" if score >= 0.75 else "🟡 MEDIUM" if score >= 0.5 else "🟢 LOW"
    print(f"  {community} ({row['region']}): {score:.2f} — {status} — {int(row['alerts'])} alerts")

print(f"\nTotal alerts triggered: {df['alert_triggered'].sum()}")

=== AgroAlert Ghana — 15 Community Alert Summary ===

  Goaso (Ahafo): 0.96 — 🔴 HIGH — 4 alerts
  Bolgatanga (Upper East): 0.91 — 🔴 HIGH — 1 alerts
  Sunyani (Bono): 0.88 — 🔴 HIGH — 3 alerts
  Damongo (Savannah): 0.88 — 🔴 HIGH — 3 alerts
  Ho (Volta): 0.88 — 🔴 HIGH — 4 alerts
  Koforidua (Eastern): 0.88 — 🔴 HIGH — 4 alerts
  Tamale (Northern): 0.88 — 🔴 HIGH — 1 alerts
  Techiman (Bono East): 0.88 — 🔴 HIGH — 2 alerts
  Nalerigu (North East): 0.88 — 🔴 HIGH — 1 alerts
  Dambai (Oti): 0.88 — 🔴 HIGH — 1 alerts
  Kumasi (Ashanti): 0.88 — 🔴 HIGH — 1 alerts
  Sekondi-Takoradi (Western): 0.88 — 🔴 HIGH — 1 alerts
  Sefwi Wiawso (Western North): 0.25 — 🟢 LOW — 0 alerts
  Wa (Upper West): 0.25 — 🟢 LOW — 0 alerts
  Cape Coast (Central): 0.18 — 🟢 LOW — 0 alerts

Total alerts triggered: 26
